In [0]:
%sql

-- Show the most recent pipeline task executions

SELECT
    run_id,
    batch_id,
    layer_name,
    status,
    input_row_count,
    output_row_count,
    start_timestamp,
    end_timestamp,
    error_message
FROM online_retail.control.pipeline_runs
ORDER BY
    COALESCE(end_timestamp, start_timestamp) DESC,
    layer_name
LIMIT 100;

run_id,batch_id,layer_name,status,input_row_count,output_row_count,start_timestamp,end_timestamp,error_message
198667464118090,2010-02,validation,SUCCESS,29388,2577,2026-09-12T09:45:17.985Z,2026-09-12T09:45:28.721Z,null
198667464118090,2010-02,gold,SUCCESS,29058,2577,2026-09-12T09:44:44.494Z,2026-09-12T09:45:07.938Z,null
198667464118090,2010-02,silver,SUCCESS,29388,29058,2026-09-12T09:44:04.147Z,2026-09-12T09:44:34.055Z,null
198667464118090,2010-02,bronze,SUCCESS,29388,29388,2026-09-12T09:43:13.466Z,2026-09-12T09:43:50.963Z,null
694459744721949,2099-01,bronze,FAILED,null,null,2026-09-12T08:36:26.561Z,2026-09-12T08:37:07.159Z,RunExecutionError
694459744721949,2099-01,gold,UPSTREAM_FAILED,null,null,null,2026-09-12T08:37:07.159Z,Skipped
694459744721949,2099-01,silver,UPSTREAM_FAILED,null,null,null,2026-09-12T08:37:07.159Z,Skipped
694459744721949,2099-01,validation,UPSTREAM_FAILED,null,null,null,2026-09-12T08:37:07.159Z,Skipped
validation-cleanup-test-001,2010-02,validation,SUCCESS,29388,2577,2026-09-12T08:19:20.469Z,2026-09-12T08:19:30.829Z,null
gold-cleanup-test-001,2010-02,gold,SUCCESS,29058,2577,2026-09-12T08:14:34.837Z,2026-09-12T08:14:56.276Z,null


In [0]:
%sql

-- Show tasks that did not complete successfully

SELECT
    run_id,
    batch_id,
    layer_name,
    status,
    start_timestamp,
    end_timestamp,
    error_message
FROM online_retail.control.pipeline_runs
WHERE status <> 'SUCCESS'
ORDER BY
    COALESCE(end_timestamp, start_timestamp) DESC,
    layer_name;

run_id,batch_id,layer_name,status,start_timestamp,end_timestamp,error_message
694459744721949,2099-01,bronze,FAILED,2026-09-12T08:36:26.561Z,2026-09-12T08:37:07.159Z,RunExecutionError
694459744721949,2099-01,gold,UPSTREAM_FAILED,null,2026-09-12T08:37:07.159Z,Skipped
694459744721949,2099-01,silver,UPSTREAM_FAILED,null,2026-09-12T08:37:07.159Z,Skipped
694459744721949,2099-01,validation,UPSTREAM_FAILED,null,2026-09-12T08:37:07.159Z,Skipped
763355269151321,2099-01,bronze,FAILED,2026-09-11T15:07:11.277Z,2026-09-11T15:07:48.636Z,RunExecutionError
763355269151321,2099-01,gold,UPSTREAM_FAILED,null,2026-09-11T15:07:48.636Z,Skipped
763355269151321,2099-01,silver,UPSTREAM_FAILED,null,2026-09-11T15:07:48.636Z,Skipped
763355269151321,2099-01,validation,UPSTREAM_FAILED,null,2026-09-11T15:07:48.636Z,Skipped
1086009018910764,2099-01,bronze,FAILED,2026-09-11T15:02:21.973Z,2026-09-11T15:02:43.013Z,RunExecutionError
1086009018910764,2099-01,gold,UPSTREAM_FAILED,null,2026-09-11T15:02:43.013Z,Skipped


In [0]:
%sql

-- Summarize the recorded task results for each pipeline run

SELECT
    run_id,
    batch_id,
    COUNT(*) AS recorded_task_count,
    SUM(
        CASE
            WHEN status = 'SUCCESS' THEN 1
            ELSE 0
        END
    ) AS successful_task_count,
    SUM(
        CASE
            WHEN status <> 'SUCCESS' THEN 1
            ELSE 0
        END
    ) AS unsuccessful_task_count,
    MIN(start_timestamp) AS pipeline_start_timestamp,
    MAX(end_timestamp) AS pipeline_end_timestamp
FROM online_retail.control.pipeline_runs
GROUP BY
    run_id,
    batch_id
ORDER BY pipeline_end_timestamp DESC;

run_id,batch_id,recorded_task_count,successful_task_count,unsuccessful_task_count,pipeline_start_timestamp,pipeline_end_timestamp
198667464118090,2010-02,4,4,0,2026-09-12T09:43:13.466Z,2026-09-12T09:45:28.721Z
694459744721949,2099-01,4,0,4,2026-09-12T08:36:26.561Z,2026-09-12T08:37:07.159Z
validation-cleanup-test-001,2010-02,1,1,0,2026-09-12T08:19:20.469Z,2026-09-12T08:19:30.829Z
gold-cleanup-test-001,2010-02,1,1,0,2026-09-12T08:14:34.837Z,2026-09-12T08:14:56.276Z
silver-cleanup-test-001,2010-02,1,1,0,2026-09-12T08:08:29.653Z,2026-09-12T08:08:55.130Z
bronze-cleanup-test-001,2010-02,1,1,0,2026-09-12T08:00:26.465Z,2026-09-12T08:00:50.997Z
117822346364439,2010-02,4,4,0,2026-09-12T07:40:32.611Z,2026-09-12T07:42:14.314Z
696155109559730,2010-02,4,4,0,2026-09-12T07:31:14.689Z,2026-09-12T07:33:33.506Z
763355269151321,2099-01,4,0,4,2026-09-11T15:07:11.277Z,2026-09-11T15:07:48.636Z
1086009018910764,2099-01,4,0,4,2026-09-11T15:02:21.973Z,2026-09-11T15:02:43.013Z
